# MFFT Training — Step by Step
**Multi-Frequency Fusion Transformer for AI Image Detection**

In [1]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# Fix project root
PROJECT_ROOT = Path(os.path.abspath('')).parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')


Project root: c:\Users\ROWTECH\Desktop\ReSearch\ai-image-detection-research
Torch: 2.12.1+cpu, CUDA: False


In [2]:
# Cell 2: Load Dataset
from src.dataset import AIDetectionDataset, ImageTransform, create_dataloaders
from src.config import Config

# Use tiny model, small batch for CPU
cfg = Config()
cfg.training.model_variant = 'tiny'
cfg.training.image_size = 224
cfg.training.batch_size = 8
cfg.training.mixed_precision = False
cfg.training.num_workers = 0
cfg.training.epochs = 1
cfg.training.val_check_interval = 50
cfg.training.gradient_accumulation_steps = 1
cfg.dataset.val_split = 0.1

# Load full dataset
full_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=[str(PROJECT_ROOT / 'dataset' / 'metadata' / 'all.csv')],
    transform=None,
    is_train=True,
    size=cfg.training.image_size,
    undersample=True,
)

print(f'Total samples: {len(full_dataset)}')
print(f'Real: {sum(1 for _, l in full_dataset.samples if l==0)}')
print(f'AI:   {sum(1 for _, l in full_dataset.samples if l==1)}')

Dataset loaded: 35806 samples
  Real: 17903, AI: 17903, Total: 35806
Total samples: 35806
Real: 17903
AI:   17903


In [3]:
# Cell 3: Split into Train/Val
from sklearn.model_selection import train_test_split

labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, val_idx = train_test_split(
    indices, test_size=cfg.dataset.val_split,
    stratify=labels, random_state=42,
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

# Build datasets with transforms
train_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=True),
    is_train=True, size=cfg.training.image_size, undersample=False,
)
train_dataset.samples = [full_dataset.samples[i] for i in train_idx]

val_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
val_dataset.samples = [full_dataset.samples[i] for i in val_idx]

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

Train: 32225, Val: 3581
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Train batches: 4029, Val batches: 448


In [4]:
# Cell 4: Build Model
from src.model import build_mfft, count_parameters, MFFTWithExplainability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = build_mfft('tiny')
model = model.to(device)
print(f'Parameters: {count_parameters(model):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
print('Model ready')

Device: cpu
Parameters: 372,450
Model ready


In [5]:
# Cell 5: Train 1 Epoch (watch progress)
model.train()
total_loss = 0
correct = 0
total = 0
start = time.time()

pbar = tqdm(train_loader, desc='Training')
for batch_idx, (images, labels) in enumerate(pbar):
    images, labels = images.to(device), labels.to(device)
    
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()
    
    optimizer.step()
    optimizer.zero_grad()
    
    total_loss += loss.item()
    preds = logits.argmax(dim=-1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)
    
    if (batch_idx + 1) % 20 == 0:
        acc = correct / total * 100
        pbar.set_postfix({'loss': f'{total_loss/(batch_idx+1):.4f}', 'acc': f'{acc:.2f}%'})

elapsed = time.time() - start
print(f'Epoch done in {elapsed:.1f}s')
print(f'Train loss: {total_loss/len(train_loader):.4f}, acc: {correct/total*100:.2f}%')

Training:   0%|          | 0/4029 [00:00<?, ?it/s]

c:\Users\ROWTECH\Desktop\ReSearch\ai-image-detection-research\.venv\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch done in 2303.2s
Train loss: 0.6902, acc: 53.80%


In [6]:
# Cell 6: Validate
model.eval()
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Validating'):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        val_loss += loss.item()
        preds = logits.argmax(dim=-1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

print(f'Val loss: {val_loss/len(val_loader):.4f}, acc: {val_correct/val_total*100:.2f}%')

Validating:   0%|          | 0/448 [00:00<?, ?it/s]

Val loss: 0.6919, acc: 52.75%


In [7]:
# Cell 7: Full Training Loop (adjust epochs)
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

NUM_EPOCHS = 5  # reduce for CPU; use 50 for full

model = build_mfft('tiny').to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

best_acc = 0
for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}', 'acc': f'{correct/total*100:.2f}%', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    train_acc = correct / total * 100
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total * 100
    print(f'Epoch {epoch+1}: train={train_acc:.2f}%, val={val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'best_mfft_tiny.pt')
        print(f'  Saved best model ({best_acc:.2f}%)')

print(f'\nBest val accuracy: {best_acc:.2f}%')

Epoch 1/5:   0%|          | 0/4029 [00:00<?, ?it/s]

Epoch 1: train=52.94%, val=49.99%
  Saved best model (49.99%)


Epoch 2/5:   0%|          | 0/4029 [00:00<?, ?it/s]

Epoch 2: train=53.59%, val=54.23%
  Saved best model (54.23%)


Epoch 3/5:   0%|          | 0/4029 [00:00<?, ?it/s]

Epoch 3: train=55.81%, val=57.94%
  Saved best model (57.94%)


Epoch 4/5:   0%|          | 0/4029 [00:00<?, ?it/s]

Epoch 4: train=58.28%, val=58.81%
  Saved best model (58.81%)


Epoch 5/5:   0%|          | 0/4029 [00:00<?, ?it/s]

Epoch 5: train=59.25%, val=59.09%
  Saved best model (59.09%)

Best val accuracy: 59.09%


In [8]:
# Cell 8: Save Final Model
model.eval()
torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'mfft_tiny_final.pt')
print('Model saved to model/checkpoints/mfft_tiny_final.pt')
print(f'File size: {os.path.getsize(PROJECT_ROOT / "model" / "checkpoints" / "mfft_tiny_final.pt") / 1e6:.1f} MB')

Model saved to model/checkpoints/mfft_tiny_final.pt
File size: 1.5 MB


In [9]:
# Cell 9: Test Prediction on a Sample
img_path = val_dataset.samples[0][0]
img = Image.open(img_path).convert('RGB')
transform = ImageTransform(size=cfg.training.image_size, augment=False)
tensor = transform(img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(tensor)
    probs = F.softmax(logits, dim=-1)

pred = 'AI-generated' if probs[0][1] > probs[0][0] else 'Real'
print(f'Image: {os.path.basename(img_path)}')
print(f'Real prob: {probs[0][0]:.4f}')
print(f'AI prob:   {probs[0][1]:.4f}')
print(f'Prediction: {pred}')

Image: PEXELS_8015785.jpg
Real prob: 0.6051
AI prob:   0.3949
Prediction: Real


---
## Manuscript Figures (Cell 10)
Generate all publication-quality figures after training.

In [12]:
# Cell 10: Generate All Manuscript Figures
from src.visualize import generate_all_figures
from sklearn.metrics import confusion_matrix
import numpy as np

FIGS_DIR = PROJECT_ROOT / 'paper' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# Gather predictions for ROC/PR curves and confusion matrix
all_val_labels = []
all_val_probs = []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=-1)
        all_val_labels.extend(labels.cpu().numpy())
        all_val_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_val_labels)
y_score = np.array(all_val_probs)
y_pred = (y_score >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)

# Class labels from dataset
all_labels = [s[1] for s in full_dataset.samples]

# Pick a sample image for frequency decomposition
sample_img = None
for cls_dir in ['real', 'ai_generated', 'ai_altered']:
    d = PROJECT_ROOT / 'dataset' / 'images' / cls_dir
    if d.exists():
        files = list(d.glob('*.jpg')) or list(d.glob('*.png'))
        if files:
            sample_img = str(files[0])
            break

# Generate all 10 figures
paths = generate_all_figures(
    model=model,
    val_loader=val_loader,
    device=device,
    y_true=y_true,
    y_score=y_score,
    cm=cm,
    labels=all_labels,
    sample_image_path=sample_img,
    output_dir=FIGS_DIR,
)

print('\n' + '='*60)
print('All manuscript figures saved to paper/figures/')
for name, path in paths.items():
    print(f'  {name}: {path.name}')
print('='*60)

Generating Figure 1: Frequency Decomposition...


IndexError: index 3 is out of bounds for axis 1 with size 3